# Merging with crisis

Country-level panels for the macro-only models (NB12). The bank-level micro-only configuration
takes its columns from the integrated panel, so no separate micro panel is built here (audit A32:
the unused micro individual merges were removed).

## Base scenarios

In [ ]:
import os
import re
import numpy as np 
import pandas as pd

### Macro

In [ ]:
# load the correct datasets

macro_set = pd.read_parquet("../data/macro/clean/parquet_saves/macro_base_1999_2017.parquet")

crisis_panel = pd.read_parquet("../data/crisis/clean/parquet_saves/crisis_panel.parquet")

In [ ]:
macro_set.head()

In [ ]:
for name, df in zip(["macro_set", "crisis_panel"], [ macro_set, crisis_panel]): print(f"{name}: {df['year'].min()} - {df['year'].max()} ({len(df)} rows)")

macro_ind_base = macro_set.merge(crisis_panel, left_on=['country_iso3', "year"],
         right_on=['country_iso3', "year"], how="left")

print("Loaded df shape:", macro_ind_base.shape)
print("First columns:", list(macro_ind_base.columns[:15]))
print(macro_ind_base.head())

In [ ]:
macro_ind_base.head()

In [ ]:
macro_ind_base = macro_ind_base.rename(columns={"country_iso3": "country_iso"})
macro_ind_base["crisis"] = macro_ind_base["crisis"].fillna(0).astype(int)
macro_ind_base.head()
print(macro_ind_base.columns.tolist())

In [ ]:

#4. Check which countries have crisis=1
macro_ind_base[macro_ind_base["crisis"] == 1][["country_iso", "year"]].drop_duplicates()
#Every country-year with crisis=1 (duration coding: all years of each episode, not only onsets).

In [ ]:

#5. No duplicate country_iso3 column lingering
print([c for c in macro_ind_base.columns if "iso3" in c])  # should be empty

In [ ]:
import os
os.makedirs("../data/individual_merges/merged/parquet_saves/macro_individual_merges", exist_ok=True)
os.makedirs("../data/individual_merges/merged/csv_saves/macro_individual_merges", exist_ok=True)
macro_ind_base.to_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_base.parquet", index=False)
macro_ind_base.to_csv("../data/individual_merges/merged/csv_saves/macro_individual_merges/macro_ind_base.csv", index=False)

## Robust scenarios

### Macro

In [ ]:
# load the correct datasets

macro_ind_r = pd.read_parquet("../data/macro/clean/parquet_saves/macro_robust_1995_2017.parquet")

crisis_panel_r = pd.read_parquet("../data/crisis/clean/parquet_saves/crisis_panel.parquet")

In [ ]:
macro_ind_r.head()

In [ ]:
for name, df in zip(["macro_ind_r", "crisis_panel_r"], [macro_ind_r, crisis_panel_r]): print(f"{name}: {df['year'].min()} - {df['year'].max()} ({len(df)} rows)")

macro_ind_robust = macro_ind_r.merge(crisis_panel_r, left_on=['country_iso3', "year"],
            right_on=['country_iso3', "year"], how="left")

print("Loaded df shape:", macro_ind_robust.shape)
print("First columns:", list(macro_ind_robust.columns[:15]))
print(macro_ind_robust.head())

In [ ]:
macro_ind_robust = macro_ind_robust.rename(columns={"country_iso3": "country_iso"})
macro_ind_robust["crisis"] = macro_ind_robust["crisis"].fillna(0).astype(int)

In [ ]:
macro_ind_robust.head()
print(macro_ind_robust.columns.tolist())

In [ ]:

#4. Check which countries have crisis=1
macro_ind_robust[macro_ind_robust["crisis"] == 1][["country_iso", "year"]].drop_duplicates()
#Every country-year with crisis=1 (duration coding: all years of each episode, not only onsets).

In [ ]:
import os
os.makedirs("../data/individual_merges/merged/parquet_saves/macro_individual_merges", exist_ok=True)
os.makedirs("../data/individual_merges/merged/csv_saves/macro_individual_merges", exist_ok=True)
macro_ind_robust.to_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_robust.parquet", index=False)
macro_ind_robust.to_csv("../data/individual_merges/merged/csv_saves/macro_individual_merges/macro_ind_robust.csv", index=False)

# Feature engineering — macro standalone only
The micro-only model comparison is handled by column selection from the integrated dataset (me_merged_base / me_merged_robust).
Only the country-level macro panels need feature engineering here — they feed NB12 (standalone macro EWS).

## Base scenarios

### Reorder columns

In [ ]:
macro_ind_base_me = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_base.parquet")
macro_ind_base_me = macro_ind_base_me.sort_values(['country_iso', 'year']).reset_index(drop=True)

In [ ]:
priority_cols_macro = ['country_iso', 'year', 'crisis']

In [ ]:
macro_ind_base_remain = [c for c in macro_ind_base_me.columns if c not in priority_cols_macro]
macro_ind_base_me = macro_ind_base_me[priority_cols_macro + macro_ind_base_remain]

macro_ind_base_me.head()

### Lagging features

base scenarios

In [ ]:
feature_cols_macro_base = [c for c in macro_ind_base_me.columns if c not in ['country_iso', 'year', 'crisis']]

In [ ]:
macro_ind_base_me[feature_cols_macro_base] = macro_ind_base_me.groupby('country_iso')[feature_cols_macro_base].shift(1)

In [ ]:
macro_ind_base_me = macro_ind_base_me.dropna(subset=feature_cols_macro_base, how='all').reset_index(drop=True)

In [ ]:
print(macro_ind_base_me.groupby('country_iso')['year'].min().head(10))

In [ ]:
# Spot-check macro base lag: NGDP_RPCH at year t in the lagged df should equal NGDP_RPCH at year t-1 in the original.
# (Crisis is NOT a feature — it is never shifted — so we must check an actual feature column here.)
_orig_macro_base = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_base.parquet")
_orig_macro_aut  = _orig_macro_base[_orig_macro_base['country_iso'] == 'AUT'][['year', 'NGDP_RPCH']].sort_values('year').reset_index(drop=True)
_lagged_macro_aut = macro_ind_base_me[macro_ind_base_me['country_iso'] == 'AUT'][['year', 'NGDP_RPCH']].sort_values('year').reset_index(drop=True)

print("Original (NGDP_RPCH at year t):")
print(_orig_macro_aut.head(6))
print("\nLagged (NGDP_RPCH at year t should equal original year t-1):")
print(_lagged_macro_aut.head(6))

In [ ]:
# Shape check: exactly one row per country should be dropped (the first year has no t-1 to shift from).
_orig_macro_base       = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_base.parquet")
n_countries_macro_base = _orig_macro_base['country_iso'].nunique()
dropped_macro_base     = len(_orig_macro_base) - len(macro_ind_base_me)

print(f"Macro base — dropped {dropped_macro_base} rows, expected {n_countries_macro_base}: {'OK' if dropped_macro_base == n_countries_macro_base else 'MISMATCH'}")

In [ ]:
import os
os.makedirs("../data/individual_merges/parquet_saves", exist_ok=True)
os.makedirs("../data/individual_merges/csv_saves", exist_ok=True)

macro_ind_base_me.to_parquet("../data/individual_merges/parquet_saves/macro_ind_base_me.parquet", index=False)
macro_ind_base_me.to_csv("../data/individual_merges/csv_saves/macro_ind_base_me.csv", index=False)

robustness scenarios

In [ ]:
macro_ind_robust_me = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_robust.parquet")
macro_ind_robust_me = macro_ind_robust_me.sort_values(['country_iso', 'year']).reset_index(drop=True)

In [ ]:
macro_ind_robust_remain = [c for c in macro_ind_robust_me.columns if c not in priority_cols_macro]
macro_ind_robust_me = macro_ind_robust_me[priority_cols_macro + macro_ind_robust_remain]

macro_ind_robust_me.head()

### Lagging features (robust)

In [ ]:
feature_cols_macro_robust = [c for c in macro_ind_robust_me.columns if c not in ['country_iso', 'year', 'crisis']]

# Shift features back by one year within each entity so the model uses t-1 data to predict crisis at t
macro_ind_robust_me[feature_cols_macro_robust] = macro_ind_robust_me.groupby('country_iso')[feature_cols_macro_robust].shift(1)

# Drop the first observation per entity — it has no t-1 features and cannot be used for training
macro_ind_robust_me = macro_ind_robust_me.dropna(subset=feature_cols_macro_robust, how='all').reset_index(drop=True)

print(macro_ind_robust_me.groupby('country_iso')['year'].min().head(10))

In [ ]:
# Spot-check: NGDP_RPCH at year t in lagged df should equal NGDP_RPCH at year t-1 in original
_orig_macro_robust  = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_robust.parquet")
_orig_macro_r_aut   = _orig_macro_robust[_orig_macro_robust['country_iso'] == 'AUT'][['year', 'NGDP_RPCH']].sort_values('year').reset_index(drop=True)
_lagged_macro_r_aut = macro_ind_robust_me[macro_ind_robust_me['country_iso'] == 'AUT'][['year', 'NGDP_RPCH']].sort_values('year').reset_index(drop=True)

print("Original NGDP_RPCH at year t:")
print(_orig_macro_r_aut.head(6))
print("\nLagged NGDP_RPCH at year t (should equal original t-1):")
print(_lagged_macro_r_aut.head(6))

# Shape check: exactly one row per country dropped
n_countries_macro_robust = _orig_macro_robust['country_iso'].nunique()
dropped_macro_robust     = len(_orig_macro_robust) - len(macro_ind_robust_me)
print(f"\nMacro robust — dropped {dropped_macro_robust} rows, expected {n_countries_macro_robust}: {'OK' if dropped_macro_robust == n_countries_macro_robust else 'MISMATCH'}")

In [ ]:
macro_ind_robust_me.to_parquet("../data/individual_merges/parquet_saves/macro_ind_robust_me.parquet", index=False)
macro_ind_robust_me.to_csv("../data/individual_merges/csv_saves/macro_ind_robust_me.csv", index=False)

### robustness t-2

Base

In [ ]:
macro_ind_base_lag2_me = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_base.parquet")
macro_ind_base_lag2_me = macro_ind_base_lag2_me.sort_values(['country_iso', 'year']).reset_index(drop=True)

In [ ]:
feature_cols_macro_base = [c for c in macro_ind_base_lag2_me.columns if c not in ['country_iso', 'year', 'crisis']]

# Shift features back by one year within each entity so the model uses t-2 data to predict crisis at t
macro_ind_base_lag2_me[feature_cols_macro_base] = macro_ind_base_lag2_me.groupby('country_iso')[feature_cols_macro_base].shift(2)

# Drop the first observation per entity — it has no t-1 features and cannot be used for training
macro_ind_base_lag2_me = macro_ind_base_lag2_me.dropna(subset=feature_cols_macro_base, how='all').reset_index(drop=True)

print(macro_ind_base_lag2_me.groupby('country_iso')['year'].min().head(10))

In [ ]:
# Spot-check macro base lag: NGDP_RPCH at year t in the lagged df should equal NGDP_RPCH at year t-2 in the original.
# (Crisis is NOT a feature — it is never shifted — so we must check an actual feature column here.)
_orig_macro_base = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_base.parquet")
_orig_macro_aut  = _orig_macro_base[_orig_macro_base['country_iso'] == 'AUT'][['year', 'NGDP_RPCH']].sort_values('year').reset_index(drop=True)
_lagged2_macro_aut = macro_ind_base_lag2_me[macro_ind_base_lag2_me['country_iso'] == 'AUT'][['year', 'NGDP_RPCH']].sort_values('year').reset_index(drop=True)

print("Original (NGDP_RPCH at year t):")
print(_orig_macro_aut.head(6))
print("\nLagged (NGDP_RPCH at year t should equal original year t-2):")
print(_lagged2_macro_aut.head(6))

In [ ]:
# Shape check: exactly one row per country should be dropped (the first two years have no t-2 to shift from).
_orig_macro_base       = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_base.parquet")
n_countries_macro_base = _orig_macro_base['country_iso'].nunique()
dropped2_macro_base     = len(_orig_macro_base) - len(macro_ind_base_lag2_me)

print(f"Macro base lag2 — dropped {dropped2_macro_base} rows, expected {n_countries_macro_base * 2}: {'OK' if dropped2_macro_base == n_countries_macro_base * 2 else 'MISMATCH'}")

In [ ]:
import os
os.makedirs("../data/individual_merges/parquet_saves", exist_ok=True)
os.makedirs("../data/individual_merges/csv_saves", exist_ok=True)

macro_ind_base_lag2_me.to_parquet("../data/individual_merges/parquet_saves/macro_ind_base_lag2_me.parquet", index=False)
macro_ind_base_lag2_me.to_csv("../data/individual_merges/csv_saves/macro_ind_base_lag2_me.csv", index=False)

Robustness

In [ ]:
macro_ind_robust_lag2_me = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_robust.parquet")
macro_ind_robust_lag2_me = macro_ind_robust_lag2_me.sort_values(['country_iso', 'year']).reset_index(drop=True)

In [ ]:
feature_cols_macro_robust = [c for c in macro_ind_robust_lag2_me.columns if c not in ['country_iso', 'year', 'crisis']]

# Shift features back by one year within each entity so the model uses t-2 data to predict crisis at t
macro_ind_robust_lag2_me[feature_cols_macro_robust] = macro_ind_robust_lag2_me.groupby('country_iso')[feature_cols_macro_robust].shift(2)

# Drop the first observation per entity — it has no t-1 features and cannot be used for training
macro_ind_robust_lag2_me = macro_ind_robust_lag2_me.dropna(subset=feature_cols_macro_robust, how='all').reset_index(drop=True)

print(macro_ind_robust_lag2_me.groupby('country_iso')['year'].min().head(10))

In [ ]:
# Spot-check macro robust lag: NGDP_RPCH at year t in the lagged df should equal NGDP_RPCH at year t-2 in the original.
# (Crisis is NOT a feature — it is never shifted — so we must check an actual feature column here.)
_orig_macro_robust = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_robust.parquet")
_orig_macro_aut  = _orig_macro_robust[_orig_macro_robust['country_iso'] == 'AUT'][['year', 'NGDP_RPCH']].sort_values('year').reset_index(drop=True)
_lagged2_macro_aut = macro_ind_robust_lag2_me[macro_ind_robust_lag2_me['country_iso'] == 'AUT'][['year', 'NGDP_RPCH']].sort_values('year').reset_index(drop=True)

print("Original (NGDP_RPCH at year t):")
print(_orig_macro_aut.head(6))
print("\nLagged (NGDP_RPCH at year t should equal original year t-2):")
print(_lagged2_macro_aut.head(6))

In [ ]:
# Shape check: exactly one row per country should be dropped (the first two years have no t-2 to shift from).
_orig_macro_robust       = pd.read_parquet("../data/individual_merges/merged/parquet_saves/macro_individual_merges/macro_ind_robust.parquet")
n_countries_macro_robust = _orig_macro_robust['country_iso'].nunique()
dropped2_macro_robust     = len(_orig_macro_robust) - len(macro_ind_robust_lag2_me)

print(f"Macro robust lag2 — dropped {dropped2_macro_robust} rows, expected {n_countries_macro_robust * 2}: {'OK' if dropped2_macro_robust == n_countries_macro_robust * 2 else 'MISMATCH'}")

In [ ]:
import os
os.makedirs("../data/individual_merges/parquet_saves", exist_ok=True)
os.makedirs("../data/individual_merges/csv_saves", exist_ok=True)

macro_ind_robust_lag2_me.to_parquet("../data/individual_merges/parquet_saves/macro_ind_robust_lag2_me.parquet", index=False)
macro_ind_robust_lag2_me.to_csv("../data/individual_merges/csv_saves/macro_ind_robust_lag2_me.csv", index=False)